#### Object Detection using SSD MobileNet

In [9]:
import sys
# !"{sys.executable}" -m pip install --user --upgrade numpy==1.26.4
!"{sys.executable}" -m pip install --user --upgrade opencv-contrib-python

  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl (46.5 MB)
  Attempting uninstall: opencv-contrib-python
    Found existing installation: opencv-contrib-python 4.7.0.72
    Uninstalling opencv-contrib-python-4.7.0.72:
      Successfully uninstalled opencv-contrib-python-4.7.0.72


  You can safely remove it manually.


In [17]:
!"{sys.executable}" -m pip install "numpy==1.26.4"

  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): still running...
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [3783 lines of output]
      + c:\Users\Ajay kumar BR\AppData\Local\Programs\Python\Python313\python.exe C:\Users\Ajay kumar BR\AppData\Local\Temp\pip-install-8westwoz\numpy_adf76af8186749e89b5122946790269a\vendored-meson\meson\meson.py setup C:\Users\Ajay kumar BR\AppData\Local\Temp\pip-install-8westwoz\numpy_adf76af8186749e89b5122946790269a C:\Users\Ajay kumar BR\AppData\Local\Temp\pip-install-8westwoz\numpy_adf76af8186749e89b5122946790269a\.mesonpy-qlwpzwov -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\Ajay kumar BR\AppData\Local\Temp\pip-install-8westwoz\numpy_adf76af8186749e89b5122946790269a\.mesonpy-qlwpzwov\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\Ajay kumar BR\AppData\Local\Temp\pip-install-8westwoz\numpy_adf76af8186749e89b5122946790269a
      Build dir:

In [10]:
import cv2
import numpy
import matplotlib.pyplot as plt

%matplotlib inline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Ajay kumar BR\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Ajay kumar BR\AppData\Roaming\Python\Python313\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Ajay kumar BR\AppData\Roaming\Python\Python313\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
modelfile = 'data/models/ssd_mobilenet_frozen_inference_graph.pb'
configfile = 'data/models/ssd_mobilenet_v2_coco_2018_03_29.pbtxt'
classfile = 'data/models/coco_class_labels.txt'

import requests
from os import path

if not path.exists(modelfile):
    print('Downloading Mobilenet SSD model...')
    url = 'https://opencv-courses.s3.us-west-2.amazonaws.com/ssd_mobilenet_frozen_inference_graph.pb'

    r = requests.get(url)

    with open(modelfile, 'wb') as f:
        f.write(r.content)

    print('ssd_mobilenet_frozen_inference_graph Download complete!')

In [ ]:
net = cv2.dnn.readNetFromTensorflow(modelfile, configfile)

In [ ]:
with open(classfile) as fp:
    labels = fp.read().split('\n')
print(sorted(labels))

In [ ]:
def detect_objects(net, img):
    dim = 300
    mean = (0, 0, 0)
    blob = cv2.dnn.blobFromImage(img, 1.0, (dim, dim), mean, True, False)
    net.setInput(blob)
    objects = net.forward()

    return objects

In [ ]:
food_img = cv2.imread('data/images/fruit-vegetable.jpg')
food_objects = detect_objects(net, food_img)

print(f'Detected {len(food_objects[0, 0])} objects (no confidence filtering).')
first_detected_obj = food_objects[0, 0, 0]
print(f'First object: {first_detected_obj}')

In [ ]:
def draw_text(img, text, x, y):
    fontface = cv2.FONT_HERSHEY_SIMPLEX
    fontscale = 0.7
    thickness = 1

    textsize = cv2.getTextSize(text, fontface, fontscale, thickness)
    dim = textsize[0]
    baseline = textsize[1]

    cv2.rectangle(img, (x, y), (x+dim[0], y + dim[1] + baseline), (0, 0, 0), cv2.FILLED)
    cv2.putText(img, text, (x, y + dim[1]), fontface, fontscale, (255, 255, 255), thickness)

In [ ]:
def draw_objects(img, objects, threshold = 0.25):
    rows = img.shape[0]
    cols = img.shape[1]

    for i in range(objects.shape[2]):
        classId = int(objects[0, 0, i, 1])
        score = float(objects[0, 0, i, 2])

        x = int(objects[0, 0, i, 3] * cols)
        y = int(objects[0, 0, i, 4] * rows)
        w = int(objects[0, 0, i, 5] * cols) - x
        h = int(objects[0, 0, i, 6] * rows) - y

        if score > threshold:
            draw_text(img, '{}'.format(labels[classId]), x, y)
            cv2.rectangle(img, (x, y), (x+w, y+h), (255, 255, 255), 2)

    mp_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return mp_img

In [ ]:
result = draw_objects(food_img.copy(), food_objects, 0.4)
plt.figure(figsize = (30, 10)); plt.imshow(result); plt.show();

In [ ]:
traffic_img = cv2.imread('data/images/traffic.jpg')
traffic_objects = detect_objects(net, traffic_img)

low = draw_objects(traffic_img.copy(), traffic_objects, 0.0)
mid = draw_objects(traffic_img.copy(), traffic_objects, 0.3)
high = draw_objects(traffic_img.copy(), traffic_objects, 0.9)

plt.figure(figsize = (20, 8))
plt.subplot(131); plt.axis('off'); plt.imshow(low); plt.title('Low confidence')
plt.subplot(132); plt.axis('off'); plt.imshow(mid); plt.title('Medium confidence')
plt.subplot(133); plt.axis('off'); plt.imshow(high); plt.title('High confidence')

#### Object Detection using YOLO V4

In [ ]:
import numpy as np
import cv2
import sys
import requests
from os import path

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (15.0, 15.0)
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 14

In [ ]:
#Initialize the parameters
objectnessThreshold = 0.5 # Objectness threshold, high values filter out low objectness score
confThreshold = 0.5       # Confidence threshold, high values filter out low confidence detections
nmsThreshold = 0.4        # Non-maximum suppression threshold, low values filter out overlapping boxes
inpWidth = 416            # Width of network's input image, larger is slower but more accurate
inpHeight = 416           # Height of network's input image, larger is slower but more accurate

In [ ]:
classesfile = 'data/models/coco_class_labels.txt'
classes = None

with open(classesfile, 'rt') as f:
    classes = f.read().rstrip('\n').split('\n')

modelConfiguration = 'data/models/yolov4.cfg'
modelWeights = 'data/models/yolov4.weights'

if not path.exists(modelWeights):
    url = 'https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v3_optimal/yolov4.weights'
    r = requests.get(url)
    print('Downloading YOLO v4 weights ...')

    with open(modelWeights, 'wb') as f:
        f.write(r.content)

    print('\nyolov4.weights Download complete!')

In [ ]:
net = cv2.dnn.readNetFromDarknet(modelConfiguration, modelWeights)

In [ ]:
def getOutputsNames(net):
    layersNames = net.getLayerNames()
    return [layersNames[i - 1] for i in net.getUnconnectedOutLayers().flatten()]

In [ ]:
FONTFACE = cv2.FONT_HERSHEY_SIMPLEX
FONTSCALE = 0.7
THICKNESS = 1

def display_text(im, text, x, y):
    """Draw text onto image at location."""
    
    # Get text size 
    textSize = cv2.getTextSize(text, FONTFACE, FONTSCALE, THICKNESS)
    dim = textSize[0]
    baseline = textSize[1]
            
    # Use text size to create a black rectangle. 
    cv2.rectangle(im, (x,y), (x + dim[0], y + dim[1] + baseline), (0,0,0), cv2.FILLED);
    # Display text inside the rectangle.
    cv2.putText(im, text, (x, y + dim[1]), FONTFACE, FONTSCALE, (0, 255, 255), THICKNESS, cv2.LINE_AA)

def display_objects(frame, outs):
    frameHeight, frameWidth = frame.shape[:2]
    classIds = []
    confidences = []
    boxes = []

    for out in outs:
        out = out.reshape(-1, out.shape[-1])
        print(f'first detection raw: {out[0]}')
        print(f'min: {out.min():.3f}, max: {out.max():.3f}')
        for detection in out:
            # print(f'detection: {detection}')
            if detection[4] > objectnessThreshold:
                scores = detection[5:]
                classId = np.argmax(scores)
                confidence = scores[classId]
                # print(f'scores: {scores}, classId: {classId}, confidence: {confidence}')
                if confidence > confThreshold:
                    center_x = int(detection[0] * frameWidth)
                    center_y = int(detection[1] * frameHeight)
                    width = int(detection[2] * frameWidth)
                    height = int(detection[3] * frameHeight)

                    left = int(center_x - width / 2)
                    top = int(center_y - height / 2)
                    classIds.append(classId)
                    confidences.append(float(confidence))
                    boxes.append([left, top, width, height])

    indices = cv2.dnn.NMSBoxes(boxes, confidences, confThreshold, nmsThreshold)
    for i in indices:
        box = boxes[i]
        left = box[0]
        top = box[1]
        width = box[2]
        height = box[3]
        cv2.rectangle(frame, (left, top), (left + width, top + height), (255, 255, 255), 2)
        label = '{}: {:.2f}'.format(classes[classIds[i]], confidences[i])
        display_text(frame, label, left, top)

In [ ]:
frame = cv2.imread('data/images/traffic.jpg')

blob = cv2.dnn.blobFromImage(frame, 1/255.0, (inpWidth, inpHeight), [0, 0, 0], 1, crop = False)
net.setInput(blob)
outs = net.forward(getOutputsNames(net))

display_objects(frame, outs)

t, _ = net.getPerfProfile()
label = 'Inference time: %.2f ms' % (t * 1000.0 / cv2.getTickFrequency())
cv2.putText(frame, label, (0, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 1, cv2.LINE_AA)
plt.imshow(frame[...,::-1])
print(label)

In [ ]:
tiny_modelConfig = 'data/models/yolov4-tiny.cfg'
tiny_modelWeights = 'data/models/yolov4-tiny.weights'
tinyOnnxModel = 'data/models/yolov4-tiny.onnx'

if not path.exists(tiny_modelWeights):
    # url = 'https://huggingface.co/Kalray/yolov4-tiny/resolve/main/yolov4-tiny.onnx'
    url = "https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.weights"

    print('Downloading YOLO v4 Tiny Model ...')
    r = requests.get(url)

    with open(tinyOnnxModel, 'wb') as f:
        f.write(r.content)

    print('yolov4-tiny.weights Download complete!')

In [ ]:
net_tiny = cv2.dnn.readNetFromDarknet(tiny_modelConfig, tiny_modelWeights)

In [ ]:
frame = cv2.imread('data/images/traffic.jpg')

blob = cv2.dnn.blobFromImage(frame, 1/255.0, (inpWidth, inpHeight), [0,0,0], 1, crop=False)
net_tiny.setInput(blob)
outs = net_tiny.forward(getOutputsNames(net_tiny))

display_objects(frame, outs)

t, _ = net_tiny.getPerfProfile()
label = 'Inference time: %.2f ms' % (t *1000.0 / cv2.getTickFrequency())
cv2.putText(frame, label, (0, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 1, cv2.LINE_AA)

plt.imshow(frame[...,::-1])
print(label)
